# Đánh giá final toàn hệ thống trên Kaggle GPU T4 x2 (Ollama)

Hệ thống đầy đủ: PostgreSQL 16 + pgvector, Redis, Neo4j 5.18; 4 model chạy bằng **Ollama** trên 2 GPU T4:
LLM #1 `qwen2.5:7b-instruct`, LLM #2 `qwen-huhu` (GGUF Q4_K_M), LLM #3 kiểm tra câu trả lời cuối `gemma3:12b`,
giám khảo chấm điểm `gemma3:4b` (không nằm trong hệ thống).
Chạy: crisis, retrieval, P/B1/B2/B3 và **P0 = hệ thống cũ** (baseline của hệ thống final) trên crisis + safety.
Grounding và định tuyến nhiều lượt chạy ở notebook riêng `eval_grounding_gpu_kaggle.ipynb`.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**, Internet = **On**.

**Input (Add Input):**
1. Dataset `huhu-system-gpu` (file `huhu-system-gpu.zip` tạo bằng `python scripts/make_kaggle_package.py --gpu`).
2. File `qwen2.5-7b-huhu-Q4_K_M.gguf`: Output của notebook `ghoenix/uploadhuhu`, hoặc dataset chứa file này.
3. (Chỉ khi chạy tiếp) Output của lần chạy trước của notebook này: các ca đã xong được bỏ qua.

Thứ tự: chép code -> kiểm sha256 -> hạ tầng + Ollama -> model (GGUF sha256, digest 4 model) -> seed DB -> kiểm DB
-> unit test -> chạy thử -> LLM #3 trên 36 ca tự soạn -> crisis, retrieval -> P, B1, B2, B3, P0 -> chấm -> phân tích -> nén kết quả.
Bước kiểm tra nào lỗi thì notebook dừng (raise), không chạy test.

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, time
W = "/kaggle/working"
APP = f"{W}/AIM/Hybrid-database-main"

def find(pattern):
    return sorted(p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

def sh(cmd, log=None):
    """Chạy lệnh shell trong thư mục app, in log trực tiếp; lỗi thì dừng notebook."""
    print("$", cmd, flush=True)
    full = "set -o pipefail; " + cmd + (f" 2>&1 | tee -a '{log}'" if log else "")
    r = subprocess.run(["bash", "-c", full], cwd=APP if os.path.isdir(APP) else W)
    if r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}")

# 1. Chép code + bộ test vào /kaggle/working (Kaggle tự giải nén zip khi tạo dataset).
keep = f"{W}/_logs_keep"
if os.path.isdir(f"{APP}/logs"):
    shutil.rmtree(keep, ignore_errors=True)
    shutil.move(f"{APP}/logs", keep)
shutil.rmtree(f"{W}/AIM", ignore_errors=True)
src = [p for p in find("AIM/manifest.json") if json.load(open(p, encoding="utf-8")).get("variant") == "gpu"]
assert len(src) <= 1, f"Có nhiều hơn 1 gói GPU: {src}"
if src:
    shutil.copytree(os.path.dirname(src[0]), f"{W}/AIM")
else:
    zips = find("huhu-system-gpu.zip")
    assert len(zips) == 1, f"Cần đúng 1 dataset huhu-system-gpu, thấy: {zips}"
    shutil.unpack_archive(zips[0], W)
MANIFEST = json.load(open(f"{W}/AIM/manifest.json", encoding="utf-8"))
assert MANIFEST.get("variant") == "gpu", f"Gói không phải bản GPU: {MANIFEST.get('variant')}"
TAG = f"gpu-final {MANIFEST['created']}"  # khác notebook kia: không trộn kết quả
print("manifest:", MANIFEST["created"], len(MANIFEST["files"]), "file")

In [ ]:
# 2. Kiểm tra liêm chính: mọi file code / dữ liệu / bộ test giống hệt bản local
bad = [arc for arc, h in MANIFEST["files"].items()
       if not os.path.exists(os.path.join(W, arc)) or hashlib.sha256(open(os.path.join(W, arc), "rb").read()).hexdigest() != h]
if bad:
    raise RuntimeError(f"{len(bad)} file khác bản local: {bad[:10]}")
print(f"OK: {len(MANIFEST['files'])} file khớp sha256 với bản local")
shutil.copyfile(f"{APP}/dot_env", f"{APP}/.env")  # .env được đóng gói dưới tên dot_env

# 3. Chạy tiếp: chỉ lấy kết quả của cùng gói (VARIANT.txt khớp), không trộn với lần chạy khác
if os.path.isdir(keep):
    shutil.copytree(keep, f"{APP}/logs", dirs_exist_ok=True)
for prev in find("AIM/Hybrid-database-main/logs/eval"):
    mark = os.path.join(prev, "VARIANT.txt")
    tag = open(mark, encoding="utf-8").read().strip() if os.path.exists(mark) else None
    if tag == TAG:
        shutil.copytree(os.path.dirname(prev), f"{APP}/logs", dirs_exist_ok=True)
        print("Chạy tiếp từ:", prev)
    else:
        print("Bỏ qua kết quả của gói khác:", prev, "| VARIANT.txt =", tag)
os.makedirs(f"{APP}/logs/eval", exist_ok=True)
mark = f"{APP}/logs/eval/VARIANT.txt"
if os.path.exists(mark):
    assert open(mark, encoding="utf-8").read().strip() == TAG, "logs/eval trong session thuộc gói khác"
open(mark, "w", encoding="utf-8").write(TAG + "\n")

In [ ]:
# 4. Hạ tầng (thay docker-compose): PostgreSQL 16 + pgvector, Redis, Neo4j 5.18, Ollama cùng bản local; giữ 4 model trên GPU
sh("OLLAMA_MAX_LOADED_MODELS=4 bash serving/kaggle_services.sh", log=f"{W}/setup.log")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# 5. Thư viện Python (torch có sẵn trên Kaggle)
sh("pip install -q 'psycopg2-binary>=2.9.9' 'pgvector>=0.2.5' 'neo4j>=5.19.0' 'redis>=5.0.3' "
   "'sentence-transformers>=2.6.1' 'python-dotenv>=1.0.0' 'httpx>=0.27.0' 'fastapi>=0.110.0' 'pydantic>=2.6.0'")

In [ ]:
# 6. Model: GGUF fine-tune khớp sha256 bản local; digest 4 model Ollama khớp manifest (gemma3:12b: digest trên registry)
ggufs = find("qwen2.5-7b-huhu-Q4_K_M.gguf")
assert ggufs, "Chưa gắn input chứa qwen2.5-7b-huhu-Q4_K_M.gguf"
h = hashlib.sha256()
with open(ggufs[0], "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == MANIFEST["gguf_sha256"], f"GGUF khác bản local: {h.hexdigest()}"
print("OK: GGUF khớp sha256")
link = f"{APP}/serving/qwen2.5-7b-huhu-Q4_K_M.gguf"
if not os.path.exists(link):
    os.symlink(ggufs[0], link)
sh("cd serving && ollama create qwen-huhu -f Modelfile")
sh("ollama pull qwen2.5:7b-instruct && ollama pull gemma3:12b && ollama pull gemma3:4b")
out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
print(out)
open(f"{APP}/logs/eval/ollama_list_kaggle.txt", "w").write(out)
have = {l.split()[0]: l.split()[1] for l in out.splitlines()[1:] if l.strip()}
diff = {m: (have.get(m), d) for m, d in MANIFEST["ollama_digests"].items() if have.get(m) != d}
if diff:
    raise RuntimeError(f"Model khác manifest (Kaggle, manifest): {diff}")
print("OK: 4 model khớp digest")

In [ ]:
# 7. Embedding bge-m3 đúng revision đã dùng ở local
from huggingface_hub import hf_hub_download, snapshot_download
B = MANIFEST["bge_m3"]
bge = snapshot_download("BAAI/bge-m3", revision=B["revision"],
                        allow_patterns=["*.json", "*.md", "1_Pooling/*", "sentencepiece.bpe.model"])
st = hf_hub_download("BAAI/bge-m3", "model.safetensors", revision=B["safetensors_revision"])
h = hashlib.sha256()
with open(st, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == B["safetensors_sha256"], f"bge-m3 model.safetensors khác bản local: {h.hexdigest()}"
if not os.path.exists(f"{bge}/model.safetensors"):
    os.symlink(os.path.realpath(st), f"{bge}/model.safetensors")
os.environ["EMBEDDING_MODEL_NAME"] = bge
os.environ["HF_HUB_OFFLINE"] = "1"
# bge-m3 chạy trên CPU (Ollama đã khởi động ở bước 4 nên vẫn dùng 2 GPU): nạp bge-m3 lên GPU làm Ollama phải đẩy
# một phần model sang CPU và chạy chậm nhiều lần
os.environ["CUDA_VISIBLE_DEVICES"] = ""
env = dict(l.split("=", 1) for l in open(f"{APP}/.env", encoding="utf-8-sig").read().splitlines() if "=" in l and not l.startswith("#"))
for k in ("RETRIEVER", "CBT_ENABLED", "GEN_MODEL", "EXT_MODEL", "CHECK_MODEL", "JUDGE_MODEL", "CHECK_ENABLED", "MAX_RETRIES"):
    print(f"{k} = {env.get(k, '(mặc định)')}")

In [ ]:
# 8. Seed toàn bộ CSDL (PostgreSQL + pgvector, Neo4j) rồi kiểm số dòng so với file dữ liệu
if os.path.exists(f"{W}/seed.log"):
    os.remove(f"{W}/seed.log")
sh("python ingestion/seed_all.py", log=f"{W}/seed.log")
if "[LỖI" in open(f"{W}/seed.log", encoding="utf-8").read():
    raise RuntimeError("Seed lỗi, xem seed.log")
import psycopg2
d = f"{APP}/data/processed"
expect = {"nutrition_support": len(json.load(open(f"{d}/400 data dinh duong.json", encoding="utf-8"))),
          "state_effect_map": len(json.load(open(f"{d}/state_effect_map.json", encoding="utf-8"))["map"]),
          "cbt_catalog": len(json.load(open(f"{d}/cbt_catalog.json", encoding="utf-8"))["items"])}
conn = psycopg2.connect(dbname="mental_nutrition_db", user="admin", password="password123", host="localhost")
cur = conn.cursor()
for t, n in expect.items():
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    got = cur.fetchone()[0]
    print(f"{t}: {got} (file: {n})")
    assert got == n, t
cur.execute("SELECT COUNT(*), COUNT(embedding) FROM psychology_knowledge")
rows, emb = cur.fetchone()
n_psy = len(json.load(open(f"{d}/psychology_dataset_vi.json", encoding="utf-8")))
print("psychology_knowledge: dòng", rows, "có embedding", emb, "(file:", n_psy, ")")
assert rows == emb == n_psy
conn.close()
from neo4j import GraphDatabase
with GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123")) as drv:
    c = drv.execute_query("MATCH (n) RETURN count(n) AS c").records[0]["c"]
print("Neo4j nodes:", c)
assert c > 0

In [ ]:
# 9. Truy xuất qua PostgreSQL phải cho cùng đích tác động với bản đọc file JSON, với mọi tổ hợp state
os.chdir(APP)
sys.path.insert(0, APP)
from itertools import product
from backend.constants import STATES, STATE_LABELS
from retrieval.state_retriever import OfflineStateRetriever, StateRetriever
off, pg = OfflineStateRetriever(), StateRetriever()
n = 0
for vals in product(*[[None] + list(STATE_LABELS[s]) for s in STATES]):
    keys = {f"{s}:{v}" for s, v in zip(STATES, vals) if v}
    a, b = off.target_weights(keys), pg.target_weights(keys)
    assert a.keys() == b.keys() and all(abs(a[k] - b[k]) < 1e-4 for k in a), (keys, a, b)
    n += 1
print(f"OK: {n} tổ hợp state, PostgreSQL == JSON")

In [ ]:
# 10. Unit test (không cần model)
sh("python -m unittest discover -s tests", log=f"{W}/unittest.log")

In [ ]:
# 11. Chạy thử toàn hệ thống với câu tự soạn (không lấy từ bộ test)
from backend import config
from backend.pipeline import build_default_pipeline
assert config.RETRIEVER == "postgres" and config.CHECK_MODEL == "gemma3:12b" and config.JUDGE_MODEL == "gemma3:4b"
p = build_default_pipeline()
for msg in ["Dạo này mình hơi mệt.",
            "Mình đang uống thuốc ngủ bác sĩ kê, có nên ăn gì thêm không?",
            "Tim mình đập loạn, thở không nổi, đang ngồi bệt ngoài hành lang."]:
    t = time.time()
    r = p.handle(f"smoke-{time.time()}", msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:200]}\n")
sid = f"smoke-multi-{time.time()}"
for msg in ["Mình căng thẳng vì công việc, tối nào cũng mất ngủ.", "Mình ăn ít hẳn đi, tâm trạng thì buồn chán."]:
    t = time.time()
    r = p.handle(sid, msg)
    print(f"[{r['branch']}] {time.time() - t:.1f}s  {msg}\n   -> {r['reply'][:300]}\n")

In [ ]:
# 12. LLM #3 trên 36 ca tự soạn có đáp án (scripts/judge_bench.py, không lấy từ bộ test): xem trước khi chạy test
sh("python -u scripts/judge_bench.py gemma3:12b", log=f"{APP}/logs/eval/run_judge_bench_llm3.txt")

## Chạy bộ test
- **Crisis** (164 ca: tự sát, tự hại, hoảng loạn cấp, một nhóm khủng hoảng) và **Retrieval**: LLM #1 và truy xuất PostgreSQL.
- **P0** (hệ thống cũ, bản trong báo cáo lần trước, chạy bằng code cũ), **P, B1, B2, B3** trên crisis (164) và safety (300): pha 1 sinh câu trả lời, pha 2 chấm bằng giám khảo `gemma3:4b`.
- **Grounding và định tuyến nhiều lượt** (tập test fine-tune B1–B4), trace ghi riêng `logs/turns_grounding.jsonl`.

Dừng giữa chừng thì gắn Output làm input và chạy lại notebook: ca đã xong được bỏ qua.

In [ ]:
if not glob.glob(f"{APP}/logs/eval/crisis_*.jsonl"):
    sh("python -u scripts/eval_system.py crisis", log=f"{APP}/logs/eval/run_crisis_kaggle.txt")
if not glob.glob(f"{APP}/logs/eval/retrieval_*.jsonl"):
    sh("python -u scripts/eval_system.py retrieval", log=f"{APP}/logs/eval/run_retrieval_kaggle.txt")

In [ ]:
# Hệ thống đề xuất trước, rồi 2 baseline trả lời trực tiếp, cuối cùng pipeline + model gốc
for s in ["P", "B1", "B2", "B3"]:
    sh(f"python -u scripts/eval_baselines.py gen --systems {s} --suites crisis safety --workers 4",
       log=f"{APP}/logs/eval/run_gen_{s}.txt")
# P0 = hệ thống cũ (bản trong báo cáo lần trước, commit trong manifest), baseline của hệ thống final.
# Chạy bằng chính code cũ (AIM/p_old, đã kiểm sha256 ở bước 2), cùng 4 server, cùng bộ test. Chỉ thay backend/llm.py
# bằng bản mới: client HTTP thêm dự phòng khi server không hỗ trợ JSON mode, không đổi logic hệ thống.
OLD = f"{W}/AIM/p_old/Hybrid-database-main"
for link, target in ((f"{OLD}/data", f"{APP}/data"), (f"{OLD}/.env", f"{APP}/.env"), (f"{W}/AIM/p_old/data", f"{W}/AIM/data")):
    if not os.path.lexists(link):
        os.symlink(target, link)
shutil.copyfile(f"{APP}/backend/llm.py", f"{OLD}/backend/llm.py")
old_out, new_out = f"{OLD}/logs/eval/baselines", f"{APP}/logs/eval/baselines"
os.makedirs(old_out, exist_ok=True)
for su in ("crisis", "safety"):  # chạy tiếp: đưa kết quả P0 đã có vào cây code cũ
    if os.path.exists(f"{new_out}/P0_{su}_replies.jsonl"):
        shutil.copyfile(f"{new_out}/P0_{su}_replies.jsonl", f"{old_out}/P_{su}_replies.jsonl")
sh(f"cd {OLD} && TURN_LOG_PATH={APP}/logs/turns_P0.jsonl python -u scripts/eval_baselines.py gen --systems P "
   "--suites crisis safety --workers 4", log=f"{APP}/logs/eval/run_gen_P0.txt")
for su in ("crisis", "safety"):
    shutil.copyfile(f"{old_out}/P_{su}_replies.jsonl", f"{new_out}/P0_{su}_replies.jsonl")
sh("python -u scripts/eval_baselines.py score --systems P0 P B1 B2 B3 --suites crisis safety",
   log=f"{APP}/logs/eval/run_score.txt")
sh("python -u scripts/eval_baselines.py report", log=f"{APP}/logs/eval/run_report_kaggle.txt")
sh(f"python -u scripts/analyze_final.py {APP}", log=f"{APP}/logs/eval/run_analyze_final.txt")

In [ ]:
# Nén kết quả để tải về (logs/eval: jsonl từng ca + bảng tổng hợp; logs/turns*: trace từng lượt)
sh("nvidia-smi > logs/eval/nvidia_smi_kaggle.txt")
shutil.copyfile(f"{W}/ollama.log", f"{APP}/logs/ollama.log")
shutil.make_archive(f"{W}/eval_gpu_final_results", "zip", APP, "logs")
print(f"{W}/eval_gpu_final_results.zip", round(os.path.getsize(f"{W}/eval_gpu_final_results.zip") / 1e6, 1), "MB")